In [ ]:
import numpy as np
import pickle
from utils import reorder_transition_matrix

In [70]:
from cProfile import label
from tokenize import group


def group_states(tm, grouping):
    """
    Group states in a Markov transition matrix according to a provided grouping.
    
    Parameters:
    -----------
    transition_matrix : numpy.ndarray
        The original transition matrix where transition_matrix[i,j] represents
        the probability of transitioning from state i to state j.
    
    grouping : list of lists
        A list where each element is a list of indices that should be grouped together.
        For example, [[0,1], [2,3,4]] would group states 0 and 1 together, and states 2, 3, and 4 together.
    
    Returns:
    --------
    numpy.ndarray
        The new transition matrix after grouping states.
    """
    n_groups = len(grouping)
    new_matrix = np.zeros((n_groups, n_groups))
    
    # For each group i
    for i, group_i in enumerate(grouping):
        # Calculate the total probability in the original group
        group_i_prob = sum(sum(tm[state, :]) for state in group_i)
        
        # For each group j
        for j, group_j in enumerate(grouping):
            # Sum up all transitions from states in group i to states in group j
            sum_transitions = 0
            for state_i in group_i:
                for state_j in group_j:
                    sum_transitions += tm[state_i, state_j]
            
            # The new transition probability is the sum of all transitions 
            # from group i to group j, normalized by the total probability in group i
            if group_i_prob > 0:  # Avoid division by zero
                new_matrix[i, j] = sum_transitions / group_i_prob
            
    return new_matrix

def string_grouping_to_indices(grouping, states_order):
    index_grouping = []
    try:
        for group in grouping:
            index_group = []
            for state in group:
                index = states_order.index(state)
                index_group.append(index)
            index_grouping.append(index_group)
        return index_grouping
    except ValueError:
        raise ValueError("Invalid state in grouping")

def get_grouping():
    fg_types_chains = {'Nup2' : 16, 'Nsp1' : 48, 'Nup100' : 16, 'Nup116' : 16, 'Nup159' : 16, 'Nup49' : 32, 'Nup57' : 32, 'Nup145' : 16, 'Nup1' : 8, 'Nup60' : 16}
    grouping = [["nuc"]]
    labels = ["nuc", "cyt"]
    
    groups_nuc_fgs = []
    groups_cyt_fgs = []
    groups_ct = []
    
    for spoke in range(8):
        group_nuc_fgs = []
        group_cyt_fgs = []
        group_ct =[]
        # nucleus fgs
        for fg_type in ["Nup2", "Nup60"]:
            group_nuc_fgs.append(f"{fg_type}_{spoke:02d}")
            group_nuc_fgs.append(f"{fg_type}_{spoke+8:02d}")
        for fg_type in ["Nup1", "Nup145"]:
            group_nuc_fgs.append(f"{fg_type}_{spoke:02d}")
        
        # central fgs
        for fg_type in ["Nup49", "Nup57"]:
            group_cyt_fgs.append(f"{fg_type}_{spoke:02d}")
            group_cyt_fgs.append(f"{fg_type}_{spoke + 8:02d}")
            group_cyt_fgs.append(f"{fg_type}_{spoke + 16:02d}")    
            group_cyt_fgs.append(f"{fg_type}_{spoke + 24:02d}")
        for fg_type in ["Nsp1"]:
            group_cyt_fgs.append(f"{fg_type}_{spoke + 16:02d}")
            group_cyt_fgs.append(f"{fg_type}_{spoke + 24:02d}")
            group_cyt_fgs.append(f"{fg_type}_{spoke + 32:02d}")
            group_cyt_fgs.append(f"{fg_type}_{spoke + 40:02d}")
        for fg_type in ["Nup145"]:
            group_cyt_fgs.append(f"{fg_type}_{spoke + 8:02d}")
        
        # cytoplasm fgs
        for fg_type in ["Nup100", "Nup159", "Nsp1", "Nup116"]:
            group_ct.append(f"{fg_type}_{spoke:02d}")
            group_ct.append(f"{fg_type}_{spoke + 8:02d}")
    
        groups_nuc_fgs.append(group_nuc_fgs)
        groups_cyt_fgs.append(group_cyt_fgs)
        groups_ct.append(group_ct)  
        
    for spoke, group in enumerate(groups_nuc_fgs):
        grouping.append(group)
        labels.append(f"nuc_fgs_{spoke}")
    for spoke, group in enumerate(groups_ct):
        grouping.append(group)
        labels.append(f"central_fgs_{spoke}")
    for spoke, group in enumerate(groups_cyt_fgs):
        grouping.append(group)
        labels.append(f"cyt_fgs_{spoke}")
    grouping.append(["cyt"])
    
    return grouping, labels     

    


In [71]:
from operator import index


with open(f"data/transition_matrices/interactions-150-180-symmetric-{100}ns.pickle", "rb") as f:
    tm, states = pickle.load(f)
    
# # Load anchor coordinates
# with open(f"data/anchor_coordinates.pickle", "rb") as f:
#     anchor_coordinates = pickle.load(f)

# # Sort anchor coordinates by Z and reorder transition matrix
# anchor_coordinates = dict(sorted(anchor_coordinates.items(), key=lambda x: x[1][2]))
# new_states = ["nuc"] + list(anchor_coordinates.keys()) + ["cyt"]
# tm = reorder_transition_matrix(tm, states, new_states)  
# print(new_states)

grouping, labels = get_grouping()
index_grouping = string_grouping_to_indices(grouping, states)
new_tm = group_states(tm, index_grouping)

with open("/cs/usr/roi.eliasian/LabFolder/Master/NPC-markov/data/transition_matrices/interactions-150-180-symmetric-grouped-100ns.pickle", "wb") as f:
    pickle.dump((new_tm, labels), f)